In [32]:
import os
os.environ.pop("PYTHONPATH", None)

import sys
sys.path = [
    p for p in sys.path
    if "/opt/software-current/" not in p
]

import regex
print("regex:", regex.__file__)

regex: /mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/regex/__init__.py


# SigLIP (Image Only)

In [33]:
import pandas as pd

df = pd.read_csv("images.csv").fillna("")
query = "laptop twee"

In [34]:
import torch
from transformers import AutoModel, AutoProcessor
from PIL import Image

MODEL_NAME = "google/siglip2-base-patch16-224"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_NAME)

import torch.nn.functional as F

def embed(image_path: str):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.inference_mode():
        output = model.get_image_features(**inputs)
        embedding = output.pooler_output
        embedding = F.normalize(embedding, p=2, dim=-1)

    return embedding.squeeze(0)

def embed_text(text: str):
    inputs = processor(text=[text],
        padding="max_length", return_tensors="pt").to(device)

    with torch.inference_mode():
        output = model.get_text_features(**inputs)
        embedding = output.pooler_output
        embedding = F.normalize(embedding, p=2, dim=-1)

    return embedding.squeeze(0)

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 16739.31it/s]


In [35]:
image_embeddings = torch.stack([
    embed(str(row["image_path"])).squeeze(0).cpu()
    for _, row in df.iterrows()
]).to(device)

text_embedding = embed_text(query).to(device)

In [36]:
scores = (image_embeddings @ text_embedding.T).squeeze().cpu().numpy()

df["score_image"] = scores

# SigLIP (Caption Only)

In [37]:
caption_embeddings = torch.stack([
    embed_text(str(caption)).cpu()
    for caption in df["caption"]
]).to(device)

In [38]:
scores = (caption_embeddings @ text_embedding.T).squeeze().cpu().numpy()

df["score_caption"] = scores

# SigLIP (Image and Caption)

In [39]:
query_embedding = embed_text(query).to(device)

image_scores = image_embeddings @ query_embedding
caption_scores = caption_embeddings @ query_embedding

# Weighted combination of the two similarity signals
combined_scores = (
    0.5 * image_scores
    + (1 - 0.5) * caption_scores
)

df["score_combined"] = scores

# Evaluation

In [ ]:
def evaluate_retrieval(df, score_col, k=10):
    ranked = df.sort_values(score_col, ascending=False)
    relevant = ranked["label"].astype(bool).to_numpy()
    n_relevant = df["label"].sum()

    top_k = relevant[:k]
    precision_k = top_k.sum() / min(k, len(df))
    recall_k = top_k.sum() / n_relevant if n_relevant else 0

    precisions = [
        relevant[:i + 1].sum() / (i + 1)
        for i in range(len(relevant))
        if relevant[i]
    ]
    ap = sum(precisions) / n_relevant if n_relevant else 0

    return {
        "method": score_col,
        f"precision@{k}": precision_k,
        f"recall@{k}": recall_k,
        "average_precision": ap,
    }

scores = ["score_image", "score_caption", "score_combined"]

evaluation = pd.DataFrame([
    evaluate_retrieval(df, col, k=10)
    for col in scores
])

display(evaluation.round(3))


,method,precision@10,recall@10,average_precision
0,score_image,0.1,0.1,0.308
1,score_caption,0.1,0.1,0.252
2,score_combined,0.1,0.1,0.252
